In [24]:
### Import knihovem ###
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
import statsmodels.formula.api as smf
import os

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

In [25]:
### Načtení zdroje dat ###
path = 'NetMonitor_last12m.xlsx'
df_raw = pd.read_excel(path)
df = df_raw.copy()
os.makedirs("tables", exist_ok=True)

### Čistění dat ###
numeric_cols = ['Visits', 'Time per visit [s]', 'Visits per real user']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].replace('-', np.nan), errors='coerce')

sum_cols = ["Real users","Views","Visits","Time [s]"]

mean_cols = ["ATS [s]","Reach","Audience share","Share of time","Time per view [s]","Time per visit [s]","Views share","Views per real user","Visits per real user"]

metric_cols = ["Real users","Views","Visits","Time [s]","ATS [s]","Reach","Audience share","Share of time","Time per view [s]","Time per visit [s]","Views share","Views per real user","Visits per real user"]

### Základní statistika ###
num_cols = ['Real users', 'Views', 'Visits', 'Time [s]', 'ATS [s]',
            'Reach', 'Audience share', 'Share of time',
            'Time per view [s]', 'Time per visit [s]',
            'Views share', 'Views per real user', 'Visits per real user']

### Oddělení souhrnného řádku 'All' ###
df_all = df[df['Media channel'] == 'All'].copy().reset_index(drop=True)
df_sites = df[df['Media channel'] != 'All'].copy().reset_index(drop=True)

### Přidání pomocných sloupců ###
df_sites['Year']  = df_sites['Month'].dt.year
df_sites['Month_num'] = df_sites['Month'].dt.month
df_sites['Month_label'] = df_sites['Month'].dt.strftime('%Y-%m')

/tmp/ipykernel_778/3926547153.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = pd.to_numeric(df[col].replace('-', np.nan), errors='coerce')


In [26]:
### Základní kontrola dat ###
display(df_sites.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Media channel         108 non-null    object        
 1   Month                 108 non-null    datetime64[ns]
 2   Main focus            108 non-null    object        
 3   Real users            108 non-null    int64         
 4   Views                 108 non-null    int64         
 5   Visits                108 non-null    float64       
 6   Time [s]              108 non-null    float64       
 7   ATS [s]               108 non-null    float64       
 8   Reach                 108 non-null    float64       
 9   Audience share        108 non-null    float64       
 10  Share of time         108 non-null    float64       
 11  Time per view [s]     108 non-null    float64       
 12  Time per visit [s]    108 non-null    float64       
 13  Views share         

None

# **Basic comparison**

In [27]:
### Základní statistika ###

df_sites[num_cols].describe().T.style.format('{:.2f}')

,count,mean,std,min,25%,50%,75%,max
Real users,108.00,3135357.70,1149326.09,798160.00,2414302.00,3159904.00,4130676.00,5215584.00
Views,108.00,139863381.63,117259958.26,12196116.00,66483413.25,88745988.50,156095265.50,392954951.00
Visits,108.00,43828981.38,39904246.01,2230157.00,16936905.00,31623339.50,55001795.25,161117506.00
Time [s],108.00,8247290607.97,8517962446.10,296844052.56,3360010032.72,4219366954.44,11089318006.80,33530886374.53
ATS [s],108.00,2218.45,1668.20,326.25,934.12,1628.00,3025.03,6586.23
Reach,108.00,0.36,0.13,0.09,0.28,0.36,0.48,0.60
Audience share,108.00,0.36,0.13,0.09,0.28,0.36,0.48,0.60
Share of time,108.00,0.01,0.01,0.00,0.00,0.00,0.01,0.03
Time per view [s],108.00,54.92,22.33,15.76,32.93,53.90,74.32,93.62
Time per visit [s],108.00,174.18,34.50,115.54,150.60,170.39,203.49,238.22


In [28]:
### TABULKA - Průměrné hodnoty hlavních indikátorů podle jednotlivých webů ###
df_avg = (
    df_sites
    .groupby("Media channel")[["Real users", "ATS [s]", "Views per real user"]]
    .mean()
    .reset_index())

df_avg["Real users"] = df_avg["Real users"] / 1e6

df_avg.to_csv('tables/t01_metrics_avg.csv', index=False, sep=';', decimal=',')

In [29]:
### TABULKA - Vývoj indikátoru ve sledovaném období podle jednotlivých webů ###
### Real users ###
df_sorted = df_sites.sort_values(['Month', 'Media channel'])

df_table = df_sorted.pivot_table(
    index='Month_label',
    columns='Media channel',
    values='Real users',
    aggfunc='sum')

df_table = df_table.reset_index()

df_table.to_csv('tables/t02_RU_time.csv', index=False, sep=";", decimal=",")

In [30]:
### TABULKA - Vývoj indikátoru ve sledovaném období podle jednotlivých webů ###
### ATS[s] ###
df_sorted = df_sites.sort_values(['Month', 'Media channel'])

df_table = df_sorted.pivot_table(
    index='Month_label',
    columns='Media channel',
    values='ATS [s]',
    aggfunc='sum')

df_table = df_table.reset_index()

df_table.to_csv('tables/t02_ATS_time.csv', index=False, sep=";", decimal=",")

In [31]:
### TABULKA - Vývoj indikátoru ve sledovaném období podle jednotlivých webů ###
### Views per real user ###
df_sorted = df_sites.sort_values(['Month', 'Media channel'])

df_table = df_sorted.pivot_table(
    index='Month_label',
    columns='Media channel',
    values='Views per real user',
    aggfunc='sum')

df_table = df_table.reset_index()

df_table.to_csv('tables/t02_VPRU_time.csv', index=False, sep=";", decimal=",")

In [32]:
### Small multiples / benchmark table ###
metrics = ["Real users", "ATS [s]", "Views per real user"]
df_sm = df_sites.copy()

web_values = df_sm[["Month","Month_label","Media channel","Main focus"] + metrics].copy()

web_values = web_values.rename(columns={
    "Real users": "Web - Real users",
    "ATS [s]": "Web - ATS [s]",
    "Views per real user": "Web - Views per real user"})

focus_avg = (
    df_sm
    .groupby(["Month", "Month_label", "Main focus"])[metrics]
    .mean()
    .reset_index()
    .rename(columns={
        "Real users": "Focus avg - Real users",
        "ATS [s]": "Focus avg - ATS [s]",
        "Views per real user": "Focus avg - Views per real user"}))

market_avg = (
    df_sm
    .groupby(["Month", "Month_label"])[metrics]
    .mean()
    .reset_index()
    .rename(columns={
        "Real users": "Market avg - Real users",
        "ATS [s]": "Market avg - ATS [s]",
        "Views per real user": "Market avg - Views per real user"}))

benchmark_table = (web_values.merge(focus_avg, on=["Month", "Month_label", "Main focus"], how="left").merge(market_avg, on=["Month", "Month_label"], how="left"))

benchmark_table = benchmark_table.sort_values(["Media channel", "Month"])

num_cols = benchmark_table.select_dtypes(include="number").columns
benchmark_table[num_cols] = benchmark_table[num_cols].round(3)

benchmark_table.to_csv("tables/t03_small_multiples_benchmark.csv", index=False, sep=";", decimal=",")

In [33]:
df_web = df_sites.groupby("Media channel").agg(
    {**{col: "sum" for col in sum_cols},
     **{col: "mean" for col in mean_cols}}
).reset_index()

df_focus = df_sites.groupby("Main focus").agg(
    {**{col: "sum" for col in sum_cols},
     **{col: "mean" for col in mean_cols}}
).reset_index()

In [34]:
### Standardizace dat pomocí Z-score podle zaměření###
zscore_cols = ["Real users","ATS [s]","Views per real user","Time per view [s]","Time per visit [s]"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_web[zscore_cols])

df_scaled = pd.DataFrame(
    X_scaled,
    columns=[f"z_{col}" for col in zscore_cols])

df_scaled.insert(0, "Media channel", df_web["Media channel"])
df_scaled.to_csv("tables/t04_zscore_media.csv", index=False, sep=";", decimal=",")


In [35]:
### Standardizace dat pomocí Z-score podle zaměření###
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_focus[zscore_cols])
df_scaled = pd.DataFrame(
    X_scaled,
    columns=[f"z_{col}" for col in zscore_cols]
)
df_scaled.insert(0, "Main focus", df_focus["Main focus"])
df_scaled

,Main focus,z_Real users,z_ATS [s],z_Views per real user,z_Time per view [s],z_Time per visit [s]
0,news,1.239169,0.632309,1.029135,0.425185,1.277545
1,sport,-1.209792,0.779354,0.325469,0.955489,-0.113502
2,tabloid,-0.029376,-1.411663,-1.354604,-1.380674,-1.164043


# **Statistical analysis**

In [56]:
num_cols2 = [
    "Real users",
    "Views",
    "Visits",
    "Time [s]",
    "ATS [s]",
    "Reach",
    "Audience share",
    "Share of time",
    "Time per view [s]",
    "Time per visit [s]",
    "Views share",
    "Views per real user",
    "Visits per real user"]

corr = df_sites[num_cols2].corr(method="spearman")

corr_table = corr.reset_index()
corr_table = corr_table.rename(columns={"index": "Metric"})
corr_table = corr_table.round(3)

corr_table.to_csv("tables/t05_spearman_correlation_matrix.csv",index=False,sep=";",decimal=",")

In [37]:
### Shapiro-Wilk test normality pro Real users ###
print('Shapiro-Wilk test normality (Real users) – per server:\n')
for site, grp in df_sites.groupby('Media channel'):
    stat, p = stats.shapiro(grp['Real users'].dropna())
    normal = '✓ normální' if p > 0.05 else '✗ nenormální'
    print(f'  {site:<15}  W={stat:.4f},  p={p:.4f}  →  {normal}')

Shapiro-Wilk test normality (Real users) – per server:

  aktualne.cz      W=0.9447,  p=0.5613  →  ✓ normální
  blesk.cz         W=0.8956,  p=0.1391  →  ✓ normální
  denik.cz         W=0.8697,  p=0.0647  →  ✓ normální
  expres.cz        W=0.8872,  p=0.1085  →  ✓ normální
  extra.cz         W=0.9185,  p=0.2740  →  ✓ normální
  idnes.cz         W=0.9622,  p=0.8144  →  ✓ normální
  novinky.cz       W=0.8968,  p=0.1443  →  ✓ normální
  sport.cz         W=0.7893,  p=0.0071  →  ✗ nenormální
  super.cz         W=0.8656,  p=0.0575  →  ✓ normální


In [38]:
### Shapiro-Wilk test normality pro ATS [s] ###
print('Shapiro-Wilk test normality (ATS [s]) – per server:\n')
for site, grp in df_sites.groupby('Media channel'):
    stat, p = stats.shapiro(grp['ATS [s]'].dropna())
    normal = '✓ normální' if p > 0.05 else '✗ nenormální'
    print(f'  {site:<15}  W={stat:.4f},  p={p:.4f}  →  {normal}')

Shapiro-Wilk test normality (ATS [s]) – per server:

  aktualne.cz      W=0.9166,  p=0.2592  →  ✓ normální
  blesk.cz         W=0.9473,  p=0.5976  →  ✓ normální
  denik.cz         W=0.9772,  p=0.9697  →  ✓ normální
  expres.cz        W=0.8874,  p=0.1091  →  ✓ normální
  extra.cz         W=0.9023,  p=0.1700  →  ✓ normální
  idnes.cz         W=0.9126,  p=0.2300  →  ✓ normální
  novinky.cz       W=0.8635,  p=0.0541  →  ✓ normální
  sport.cz         W=0.9644,  p=0.8449  →  ✓ normální
  super.cz         W=0.9457,  p=0.5752  →  ✓ normální


In [39]:
### Shapiro-Wilk test normality pro Views per real user ###
print('Shapiro-Wilk test normality (Views per real user) – per server:\n')
for site, grp in df_sites.groupby('Media channel'):
    stat, p = stats.shapiro(grp['Views per real user'].dropna())
    normal = '✓ normální' if p > 0.05 else '✗ nenormální'
    print(f'  {site:<15}  W={stat:.4f},  p={p:.4f}  →  {normal}')

Shapiro-Wilk test normality (Views per real user) – per server:

  aktualne.cz      W=0.9103,  p=0.2152  →  ✓ normální
  blesk.cz         W=0.9634,  p=0.8312  →  ✓ normální
  denik.cz         W=0.8933,  p=0.1301  →  ✓ normální
  expres.cz        W=0.8755,  p=0.0767  →  ✓ normální
  extra.cz         W=0.9194,  p=0.2810  →  ✓ normální
  idnes.cz         W=0.9754,  p=0.9583  →  ✓ normální
  novinky.cz       W=0.9223,  p=0.3058  →  ✓ normální
  sport.cz         W=0.8772,  p=0.0807  →  ✓ normální
  super.cz         W=0.9553,  p=0.7154  →  ✓ normální


In [40]:
### Kruskal-Wallis test pro Real users ###
groups1 = [grp['Real users'].dropna().values
          for _, grp in df_sites.groupby('Main focus')]

h_stat, p_kw = stats.kruskal(*groups1)

print(f'Kruskal-Wallis H: {h_stat:.4f},  p = {p_kw:.4f}')
if p_kw < 0.05:
    print('→ Statisticky významný rozdíl mezi kategoriemi (p < 0.05).')
else:
    print('→ Bez statisticky významného rozdílu.')

Kruskal-Wallis H: 60.3905,  p = 0.0000
→ Statisticky významný rozdíl mezi kategoriemi (p < 0.05).


In [41]:
### Kruskal-Wallis test pro ATS [s] ###
groups2 = [grp['ATS [s]'].dropna().values
          for _, grp in df_sites.groupby('Main focus')]

h_stat, p_kw = stats.kruskal(*groups2)

print(f'Kruskal-Wallis H: {h_stat:.4f},  p = {p_kw:.4f}')
if p_kw < 0.05:
    print('→ Statisticky významný rozdíl mezi kategoriemi (p < 0.05).')
else:
    print('→ Bez statisticky významného rozdílu.')

Kruskal-Wallis H: 15.9824,  p = 0.0003
→ Statisticky významný rozdíl mezi kategoriemi (p < 0.05).


In [42]:
### Kruskal-Wallis test pro Views per real user ###
groups3 = [grp['Views per real user'].dropna().values
          for _, grp in df_sites.groupby('Main focus')]

h_stat, p_kw = stats.kruskal(*groups3)

print(f'Kruskal-Wallis H: {h_stat:.4f},  p = {p_kw:.4f}')
if p_kw < 0.05:
    print('→ Statisticky významný rozdíl mezi kategoriemi (p < 0.05).')
else:
    print('→ Bez statisticky významného rozdílu.')

Kruskal-Wallis H: 9.6213,  p = 0.0081
→ Statisticky významný rozdíl mezi kategoriemi (p < 0.05).


In [43]:
# Souhrnná tabulka: Shapiro-Wilk + Kruskal-Wallis

metrics = ["Real users", "ATS [s]", "Views per real user"]

rows = []

for metric in metrics:
    shapiro_p = df_sites.groupby("Media channel")[metric].apply(
        lambda x: stats.shapiro(x.dropna())[1]    )

    non_normal = shapiro_p[shapiro_p <= 0.05].index.tolist()

    groups = [grp[metric].dropna().values for _, grp in df_sites.groupby("Main focus")]

    h, p_kw = stats.kruskal(*groups)

    rows.append({
        "Metrika": metric,
        "Shapiro normalita": "OK" if len(non_normal) == 0 else "Problém",
        "Nenormální weby": ", ".join(non_normal) if non_normal else "-",
        "Min. Shapiro p": round(shapiro_p.min(), 4),
        "Kruskal H": round(h, 4),
        "Kruskal p": round(p_kw, 4),
        "Výsledek": "Významný rozdíl" if p_kw < 0.05 else "Bez významného rozdílu"})

summary_table = pd.DataFrame(rows)

summary_table.to_csv("tables/t06_tests_summary.csv",index=False)

summary_table

,Metrika,Shapiro normalita,Nenormální weby,Min. Shapiro p,Kruskal H,Kruskal p,Výsledek
0,Real users,Problém,sport.cz,0.0071,60.3905,0.0000,Významný rozdíl
1,ATS [s],OK,-,0.0541,15.9824,0.0003,Významný rozdíl
2,Views per real user,OK,-,0.0767,9.6213,0.0081,Významný rozdíl


In [44]:
### Čas strávený na webu podle typu webu###
model_A = smf.ols('Q("ATS [s]") ~ C(Q("Media channel"))', data=df_sites).fit()
print(model_A.summary())

### Data pro boxplot ###
df_box = df_sites[["Media channel","ATS [s]"]].copy()
df_box.to_csv("tables/t07_boxplot_ats.csv",index=False, sep=";",decimal=",")

                            OLS Regression Results                            
Dep. Variable:           Q("ATS [s]")   R-squared:                       0.985
Model:                            OLS   Adj. R-squared:                  0.984
Method:                 Least Squares   F-statistic:                     816.4
Date:                Wed, 06 May 2026   Prob (F-statistic):           9.04e-87
Time:                        13:55:05   Log-Likelihood:                -727.02
No. Observations:                 108   AIC:                             1472.
Df Residuals:                      99   BIC:                             1496.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [45]:
### Hloubka prohlížení jednotlivých webů ###
model_B_detail = smf.ols(
    'Q("Views per real user") ~ C(Q("Media channel"), Treatment(reference="novinky.cz"))',
    data=df_sites
).fit()

print(model_B_detail.summary())

                               OLS Regression Results                               
Dep. Variable:     Q("Views per real user")   R-squared:                       0.941
Model:                                  OLS   Adj. R-squared:                  0.936
Method:                       Least Squares   F-statistic:                     195.8
Date:                      Wed, 06 May 2026   Prob (F-statistic):           3.95e-57
Time:                              13:55:08   Log-Likelihood:                -342.67
No. Observations:                       108   AIC:                             703.3
Df Residuals:                            99   BIC:                             727.5
Df Model:                                 8                                         
Covariance Type:                  nonrobust                                         
                                                                              coef    std err          t      P>|t|      [0.025      0.975]
----------

In [46]:
### Hloubka prohlížení Webů Borgis oproti konkurenci ###
df_sites["Borgis"] = df_sites["Media channel"].isin(
    ["novinky.cz", "super.cz", "sport.cz"]
).astype(int)

model_borgis = smf.ols(
    'Q("Views per real user") ~ Borgis',
    data=df_sites
).fit()

print(model_borgis.summary())

                               OLS Regression Results                               
Dep. Variable:     Q("Views per real user")   R-squared:                       0.096
Model:                                  OLS   Adj. R-squared:                  0.087
Method:                       Least Squares   F-statistic:                     11.20
Date:                      Wed, 06 May 2026   Prob (F-statistic):            0.00113
Time:                              13:55:10   Log-Likelihood:                -489.68
No. Observations:                       108   AIC:                             983.4
Df Residuals:                           106   BIC:                             988.7
Df Model:                                 1                                         
Covariance Type:                  nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------

In [47]:
### Engagement index ###
engagement_metrics = ["ATS [s]","Time per view [s]","Time per visit [s]","Views per real user","Visits per real user"]

engagement_scaled = scaler.fit_transform(df_web[engagement_metrics])

df_web["Engagement index"] = engagement_scaled.mean(axis=1)

engagement_table = df_web[["Media channel", "Engagement index"]] \
    .sort_values("Engagement index", ascending=False)

In [48]:
### Size index ###
size_metrics = ["Real users","Views","Visits","Time [s]","Reach","Audience share","Views share","Share of time"]

size_scaled = scaler.fit_transform(df_web[size_metrics])

df_web["Size index"] = size_scaled.mean(axis=1)

size_table = df_web[["Media channel", "Size index"]].sort_values("Size index",ascending=False)

In [49]:
### Spojení obou indexů
comparison_table = df_web[["Media channel","Engagement index","Size index"]].copy()

comparison_table = comparison_table.sort_values("Size index", ascending=False)

comparison_table.to_csv("tables/t08_index_comparison.csv", index=False, sep=";", decimal=",")

In [50]:
# model: očekávaný ATS podle velikosti webu
model_ats_size = smf.ols(
    'Q("ATS [s]") ~ Q("Size index")',
    data=df_web
).fit()

print(model_ats_size.summary())

# predikce a rezidua
df_web_perf = df_web.copy()

df_web_perf["Predicted ATS [s]"] = model_ats_size.predict(df_web_perf)
df_web_perf["ATS residual [s]"] = (
    df_web_perf["ATS [s]"] - df_web_perf["Predicted ATS [s]"])

# standardizované reziduum
df_web_perf["ATS residual z"] = (
    df_web_perf["ATS residual [s]"] / df_web_perf["ATS residual [s]"].std())

# výsledná tabulka
ats_overperformance = (
    df_web_perf[[
        "Media channel",
        "Size index",
        "ATS [s]",
        "Predicted ATS [s]",
        "ATS residual [s]",
        "ATS residual z"
    ]]
    .sort_values("ATS residual z", ascending=False))

# export
ats_overperformance.round(3).to_csv(
    "tables/t09_ats_overperformance_by_size.csv", index=False, sep=";", decimal=",")

ats_overperformance

                            OLS Regression Results                            
Dep. Variable:           Q("ATS [s]")   R-squared:                       0.805
Model:                            OLS   Adj. R-squared:                  0.777
Method:                 Least Squares   F-statistic:                     28.93
Date:                Wed, 06 May 2026   Prob (F-statistic):            0.00103
Time:                        13:55:16   Log-Likelihood:                -72.076
No. Observations:                   9   AIC:                             148.2
Df Residuals:                       7   BIC:                             148.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept        2218.4458    274.937     

,Media channel,Size index,ATS [s],Predicted ATS [s],ATS residual [s],ATS residual z
7,sport.cz,-0.190441,2886.125916,1912.186723,973.939193,1.262333
6,novinky.cz,1.928555,6066.673123,5319.861084,746.812039,0.967951
1,blesk.cz,-0.654198,1618.246111,1166.393874,451.852237,0.585650
8,super.cz,0.327701,3083.196760,2745.439411,337.757349,0.437771
3,expres.cz,-1.323882,387.593443,89.438621,298.154822,0.386442
0,aktualne.cz,-0.428399,1320.322916,1529.513937,-209.191021,-0.271135
4,extra.cz,-0.674710,824.273135,1133.407413,-309.134277,-0.400672
5,idnes.cz,0.961794,2828.558334,3765.159760,-936.601426,-1.213939
2,denik.cz,0.053581,951.022754,2304.611670,-1353.588916,-1.754401


In [51]:
# model: očekávané Views per real user podle velikosti webu
model_views_size = smf.ols(
    'Q("Views per real user") ~ Q("Size index")',
    data=df_web
).fit()

print(model_views_size.summary())

# predikce a rezidua
df_web_perf_views = df_web.copy()

df_web_perf_views["Predicted Views per real user"] = model_views_size.predict(df_web_perf_views)

df_web_perf_views["Views residual"] = (
    df_web_perf_views["Views per real user"]
    - df_web_perf_views["Predicted Views per real user"])

# standardizované reziduum
df_web_perf_views["Views residual z"] = (
    df_web_perf_views["Views residual"]
    / df_web_perf_views["Views residual"].std())

# výsledná tabulka
views_overperformance = (
    df_web_perf_views[[
        "Media channel",
        "Size index",
        "Views per real user",
        "Predicted Views per real user",
        "Views residual",
        "Views residual z"
    ]]
    .sort_values("Views residual z", ascending=False))

# export
views_overperformance.round(3).to_csv("t10_views_overperformance_by_size.csv", index=False, sep=";", decimal=",")

views_overperformance

                               OLS Regression Results                               
Dep. Variable:     Q("Views per real user")   R-squared:                       0.602
Model:                                  OLS   Adj. R-squared:                  0.545
Method:                       Least Squares   F-statistic:                     10.59
Date:                      Wed, 06 May 2026   Prob (F-statistic):             0.0140
Time:                              13:55:18   Log-Likelihood:                -36.836
No. Observations:                         9   AIC:                             77.67
Df Residuals:                             7   BIC:                             78.07
Df Model:                                 1                                         
Covariance Type:                  nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------

,Media channel,Size index,Views per real user,Predicted Views per real user,Views residual,Views residual z
5,idnes.cz,0.961794,88.144357,58.503577,29.640780,1.927669
4,extra.cz,-0.674710,35.596766,26.768155,8.828610,0.574163
1,blesk.cz,-0.654198,35.045072,27.165928,7.879144,0.512415
7,sport.cz,-0.190441,42.857664,36.159195,6.698469,0.435631
3,expres.cz,-1.323882,12.798656,14.179285,-1.380629,-0.089788
8,super.cz,0.327701,41.448588,46.207110,-4.758522,-0.309467
6,novinky.cz,1.928555,66.326879,77.251200,-10.924321,-0.710456
0,aktualne.cz,-0.428399,17.837938,31.544671,-13.706734,-0.891409
2,denik.cz,0.053581,18.614517,40.891316,-22.276798,-1.448757


In [52]:
### Vývoj overperformance v čase (ATS + Views per real user) ###

df_over_time = df_sites.copy()

# přidání Size indexu
df_over_time = df_over_time.merge(
    df_web[["Media channel", "Size index"]],
    on="Media channel",
    how="left")

# ===== ATS =====
df_over_time["Predicted ATS [s]"] = model_ats_size.predict(df_over_time)
df_over_time["ATS residual [s]"] = (
    df_over_time["ATS [s]"] - df_over_time["Predicted ATS [s]"])

df_over_time["ATS residual z"] = (
    df_over_time["ATS residual [s]"] / df_over_time["ATS residual [s]"].std())

# ===== Views per real user =====
df_over_time["Predicted Views per real user"] = model_views_size.predict(df_over_time)

df_over_time["Views residual"] = (
    df_over_time["Views per real user"] - df_over_time["Predicted Views per real user"])

df_over_time["Views residual z"] = (
    df_over_time["Views residual"] / df_over_time["Views residual"].std())

# finální tabulka
df_over_time = df_over_time[[
    "Month_label",
    "Media channel",
    "ATS residual [s]",
    "ATS residual z",
    "Views residual",
    "Views residual z"
]].sort_values(["Media channel", "Month_label"])

# export
df_over_time.to_csv("tables/t10_overperformance_time.csv", index=False, sep=";", decimal=",")

df_over_time

,Month_label,Media channel,ATS residual [s],ATS residual z,Views residual,Views residual z
0,2025-05,aktualne.cz,-100.752529,-0.132796,-9.457349,-0.603199
8,2025-06,aktualne.cz,-78.400716,-0.103335,-11.330741,-0.722686
16,2025-07,aktualne.cz,-139.077589,-0.183310,-12.484843,-0.796296
24,2025-08,aktualne.cz,-90.990781,-0.119929,-12.422542,-0.792322
32,2025-09,aktualne.cz,-317.925800,-0.419039,-15.151456,-0.966375
...,...,...,...,...,...,...
63,2025-12,super.cz,320.759881,0.422774,-5.964893,-0.380447
71,2026-01,super.cz,574.719458,0.757503,-1.562007,-0.099626
79,2026-02,super.cz,184.077179,0.242621,-6.689356,-0.426654
87,2026-03,super.cz,338.059445,0.445576,-4.424540,-0.282202
